# Task 4: tiled attention with online softmax

This notebook contains its source code. No ZIP, dataset, adapter or model
weights need uploading. It runs standalone attention correctness tests and
performance measurements; no training takes place.

Choose **Runtime > Change runtime type > GPU** for measured GPU memory.
CPU also works, with CPU memory reported as unmeasured. Run all three steps
in order and download the result ZIP. Optimized SDPA may be faster than
the educational tiled implementation; observed slowdowns are retained.


## 1. Restore and verify the embedded source files


In [ ]:
import hashlib
from pathlib import Path

PROJECT = Path("/content/trialgpt_attention")
SOURCES = {'attention.py': '"""Inference-only dense and tiled scaled dot-product attention.\n\nImplements stable row normalization and online softmax explicitly. Inputs are\nfinite Q/K/V tensors of shape [batch, heads, tokens, head_dim]. FP16/BF16 inputs\naccumulate in FP32; FP64 precision is preserved. See ATTENTION.md for equations.\n"""\n\nimport math\n\nimport torch\n\n\ndef _prepare(q, k, v, causal, query_offset):\n    if any(x.ndim != 4 or min(x.shape) < 1 for x in (q, k, v)):\n        raise ValueError(\'Expected nonempty [batch, heads, tokens, head_dim] tensors.\')\n    if q.dtype not in (torch.float16, torch.bfloat16, torch.float32, torch.float64):\n        raise ValueError(\'Attention requires a supported floating-point dtype.\')\n    if len({x.dtype for x in (q, k, v)}) != 1 or len({x.device for x in (q, k, v)}) != 1:\n        raise ValueError(\'Q, K and V must share dtype and device.\')\n    if q.shape[0] != k.shape[0] or k.shape[:3] != v.shape[:3] or q.shape[-1] != k.shape[-1]:\n        raise ValueError(\'Incompatible batch, K/V sequence/head, or Q/K head dimensions.\')\n    if q.shape[1] % k.shape[1]:\n        raise ValueError(\'Query heads must be an integer multiple of KV heads.\')\n    if type(causal) is not bool or type(query_offset) is not int:\n        raise ValueError(\'causal must be bool and query_offset must be an integer.\')\n    accumulation = torch.float64 if q.dtype == torch.float64 else torch.float32\n    return q.to(accumulation), k.to(accumulation), v.to(accumulation)\n\n\ndef _mask(scores, q_start, k_start, query_offset):\n    rows = torch.arange(q_start, q_start + scores.shape[-2], device=scores.device) + query_offset\n    columns = torch.arange(k_start, k_start + scores.shape[-1], device=scores.device)\n    return scores.masked_fill(columns[None, :] > rows[:, None], float(\'-inf\'))\n\n\n@torch.inference_mode()\ndef reference_attention(q, k, v, *, causal=True, query_offset=0):\n    """Dense reference with scaling, causal masking and stable explicit softmax.\n\n    Query row i has absolute position i+query_offset. A fully masked row returns\n    zeros. Noncausal mode allows all keys. No dropout or attention bias is applied.\n    """\n    dtype = q.dtype\n    q, k, v = _prepare(q, k, v, causal, query_offset)\n    groups = q.shape[1] // k.shape[1]\n    if groups != 1:\n        k = k.repeat_interleave(groups, dim=1)\n        v = v.repeat_interleave(groups, dim=1)\n    scores = (q @ k.transpose(-1, -2)) * (1.0 / math.sqrt(q.shape[-1]))\n    if causal:\n        scores = _mask(scores, 0, 0, query_offset)\n    maximum = scores.amax(dim=-1, keepdim=True)\n    safe_maximum = torch.where(torch.isfinite(maximum), maximum, torch.zeros_like(maximum))\n    weights = torch.exp(scores - safe_maximum)\n    denominator = weights.sum(dim=-1, keepdim=True)\n    weights = weights / torch.where(denominator > 0, denominator, torch.ones_like(denominator))\n    return (weights @ v).to(dtype)\n\n\n@torch.inference_mode()\ndef tiled_attention(q, k, v, *, causal=True, query_offset=0, block_q=64, block_k=64):\n    """Blocked attention using running maxima, sums and weighted-value numerators.\n\n    The largest score tensor is a query/key tile, never the complete QK matrix\n    unless the chosen tile itself covers the whole input. K/V head expansion for\n    GQA happens per tile. Returned dtype equals Q.dtype. No backward pass supplied.\n    """\n    if any(type(size) is not int or size < 1 for size in (block_q, block_k)):\n        raise ValueError(\'Tile sizes must be positive integers.\')\n    dtype = q.dtype\n    q, k, v = _prepare(q, k, v, causal, query_offset)\n    batch, heads, queries, dimension = q.shape\n    keys, value_dimension = k.shape[2], v.shape[-1]\n    groups = heads // k.shape[1]\n    scale = 1.0 / math.sqrt(dimension)\n    result = torch.empty((batch, heads, queries, value_dimension), dtype=q.dtype, device=q.device)\n    for q_start in range(0, queries, block_q):\n        q_end = min(q_start + block_q, queries)\n        query_tile = q[:, :, q_start:q_end]\n        row_shape = (batch, heads, q_end - q_start, 1)\n        maximum = torch.full(row_shape, float(\'-inf\'), dtype=q.dtype, device=q.device)\n        denominator = torch.zeros(row_shape, dtype=q.dtype, device=q.device)\n        numerator = torch.zeros((*row_shape[:-1], value_dimension), dtype=q.dtype, device=q.device)\n        for k_start in range(0, keys, block_k):\n            # Every later key tile is future-only for this complete query tile.\n            if causal and k_start > q_end - 1 + query_offset:\n                break\n            k_end = min(k_start + block_k, keys)\n            key_tile, value_tile = k[:, :, k_start:k_end], v[:, :, k_start:k_end]\n            if groups != 1:\n                key_tile = key_tile.repeat_interleave(groups, dim=1)\n                value_tile = value_tile.repeat_interleave(groups, dim=1)\n            scores = (query_tile @ key_tile.transpose(-1, -2)) * scale\n            if causal:\n                scores = _mask(scores, q_start, k_start, query_offset)\n            new_maximum = torch.maximum(maximum, scores.amax(dim=-1, keepdim=True))\n            # Initially/all-masked rows have maximum=-inf; avoid -inf - -inf.\n            safe_maximum = torch.where(torch.isfinite(new_maximum), new_maximum,\n                                       torch.zeros_like(new_maximum))\n            rescale = torch.exp(maximum - safe_maximum)\n            weights = torch.exp(scores - safe_maximum)\n            denominator = rescale * denominator + weights.sum(dim=-1, keepdim=True)\n            numerator = rescale * numerator + weights @ value_tile\n            maximum = new_maximum\n        result[:, :, q_start:q_end] = numerator / torch.where(\n            denominator > 0, denominator, torch.ones_like(denominator))\n    return result.to(dtype)\n', 'benchmark_attention.py': '"""Task 4 controlled attention correctness, timing and GPU-memory experiment."""\n\nimport argparse\nimport gc\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport platform\nimport time\n\nimport torch\nimport torch.nn.functional as F\n\nfrom attention import reference_attention, tiled_attention\nfrom attention_benchmark_report import METHODS, aggregate_measurements, score_workspace_bytes, write_reports\n\n\nROOT = Path(__file__).resolve().parent\n\n\ndef write_json(path, value):\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\n\', encoding=\'utf-8\')\n    temporary.replace(path)\n\n\ndef error_metrics(actual, expected, *, atol, rtol):\n    actual, expected = actual.detach().cpu().double(), expected.detach().cpu().double()\n    if actual.shape != expected.shape:\n        raise ValueError(\'Attention output shape differs from the oracle.\')\n    finite = bool(torch.isfinite(actual).all() and torch.isfinite(expected).all())\n    if not finite:\n        return {\'finite\': False, \'passed\': False, \'max_absolute_error\': None,\n                \'mean_absolute_error\': None, \'max_relative_error\': None}\n    difference = (actual - expected).abs()\n    return {\'finite\': True, \'passed\': bool(torch.allclose(actual, expected, atol=atol, rtol=rtol)),\n            \'max_absolute_error\': difference.max().item(), \'mean_absolute_error\': difference.mean().item(),\n            \'max_relative_error\': (difference / expected.abs().clamp_min(1e-12)).max().item()}\n\n\n@torch.inference_mode()\ndef sdpa_attention(q, k, v, *, causal):\n    return F.scaled_dot_product_attention(q, k, v, is_causal=causal, dropout_p=0.0)\n\n\ndef measure(function, device):\n    gc.collect()\n    gpu = device.type == \'cuda\'\n    if gpu:\n        torch.cuda.synchronize(device)\n        torch.cuda.empty_cache()\n        torch.cuda.reset_peak_memory_stats(device)\n    initial = torch.cuda.memory_allocated(device) if gpu else None\n    started = time.perf_counter()\n    output = function()\n    if gpu:\n        torch.cuda.synchronize(device)\n    seconds = time.perf_counter() - started\n    result = {\'seconds\': seconds, \'initial_allocated_bytes\': initial,\n              \'peak_allocated_bytes\': torch.cuda.max_memory_allocated(device) if gpu else None,\n              \'peak_reserved_bytes\': torch.cuda.max_memory_reserved(device) if gpu else None}\n    del output\n    return result\n\n\ndef run_case(config, length, causal, device, seed):\n    generator = torch.Generator().manual_seed(seed)\n    dtype = getattr(torch, config[\'dtype\'])\n    shape = (config[\'batch\'], config[\'heads\'], length, config[\'head_dim\'])\n    # Generate once on CPU, then transfer; every implementation gets these tensors.\n    host = tuple(torch.randn(shape, generator=generator, dtype=dtype) for _ in range(3))\n    oracle = reference_attention(*(x.double() for x in host), causal=causal)\n    q, k, v = (x.to(device) for x in host)\n    functions = {\n        \'reference\': lambda: reference_attention(q, k, v, causal=causal),\n        \'tiled\': lambda: tiled_attention(q, k, v, causal=causal,\n            block_q=config[\'block_q\'], block_k=config[\'block_k\']),\n        \'sdpa_auto\': lambda: sdpa_attention(q, k, v, causal=causal),\n    }\n    numerical = {name: error_metrics(function(), oracle, atol=config[\'atol\'], rtol=config[\'rtol\'])\n                 for name, function in functions.items()}\n    for function in functions.values():\n        for _ in range(config[\'warmups\']):\n            function()\n    if device.type == \'cuda\':\n        torch.cuda.synchronize(device)\n    records = []\n    for repeat in range(config[\'repetitions\']):\n        order = METHODS[repeat % len(METHODS):] + METHODS[:repeat % len(METHODS)]\n        for name in order:\n            records.append({\'method\': name, \'repeat\': repeat, **measure(functions[name], device)})\n    summary = aggregate_measurements(records, repetitions=config[\'repetitions\'])\n    passed = summary[\'complete\'] and all(value[\'passed\'] for value in numerical.values())\n    if not passed:\n        summary[\'reference_over_tiled_speedup\'] = None\n    return {\'case_id\': f"n{length}-{\'causal\' if causal else \'full\'}", \'length\': length,\n            \'causal\': causal, \'seed\': seed, \'shape\': list(shape), \'passed\': passed,\n            \'numerical\': numerical, \'timing\': summary, \'measurements\': records,\n            \'score_workspace\': score_workspace_bytes(batch=config[\'batch\'], heads=config[\'heads\'],\n                queries=length, keys=length, block_q=config[\'block_q\'], block_k=config[\'block_k\'],\n                element_bytes=8 if dtype == torch.float64 else 4)}\n\n\ndef validate_config(config):\n    for key in (\'batch\', \'heads\', \'head_dim\', \'block_q\', \'block_k\', \'repetitions\', \'cpu_threads\'):\n        if type(config[key]) is not int or config[key] < 1:\n            raise ValueError(f\'{key} must be a positive integer.\')\n    lengths = config[\'lengths\']\n    if not isinstance(lengths, list) or not lengths or any(type(n) is not int or n < 1 for n in lengths) or len(set(lengths)) != len(lengths):\n        raise ValueError(\'lengths must be a nonempty list of distinct positive integers.\')\n    modes = config[\'causal_modes\']\n    if not isinstance(modes, list) or not modes or any(type(m) is not bool for m in modes) or len(set(modes)) != len(modes):\n        raise ValueError(\'causal_modes must be a nonempty list of distinct bool values.\')\n    if type(config[\'warmups\']) is not int or config[\'warmups\'] < 0:\n        raise ValueError(\'warmups must be a nonnegative integer.\')\n    for key in (\'atol\', \'rtol\'):\n        if type(config[key]) not in (int, float) or not math.isfinite(config[key]) or config[key] < 0:\n            raise ValueError(\'Numerical tolerances must be finite nonnegative numbers.\')\n    if config[\'dtype\'] not in (\'float32\', \'float64\', \'float16\', \'bfloat16\'):\n        raise ValueError(\'Unsupported benchmark dtype.\')\n\n\ndef run(args):\n    config = json.loads((args.root / \'configs/attention_config.json\').read_text(encoding=\'utf-8\'))\n    validate_config(config)\n    device = torch.device(\'cuda:0\' if args.device == \'auto\' and torch.cuda.is_available()\n                          else \'cpu\' if args.device == \'auto\' else args.device)\n    if device.type == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA is unavailable. Select a Colab GPU or use --device cpu.\')\n    torch.set_num_threads(config[\'cpu_threads\'])\n    torch.set_float32_matmul_precision(\'highest\')\n    torch.manual_seed(config[\'seed\'])\n    identity = {\'config\': config, \'device\': str(device), \'torch\': torch.__version__,\n                \'cuda\': torch.version.cuda, \'platform\': platform.platform(),\n                \'processor\': platform.processor(), \'float32_matmul_precision\': torch.get_float32_matmul_precision(),\n                \'gpu\': torch.cuda.get_device_name(device) if device.type == \'cuda\' else None,\n                \'source_sha256\': {name: hashlib.sha256((args.root / name).read_bytes()).hexdigest()\n                    for name in (\'attention.py\', \'benchmark_attention.py\', \'attention_benchmark_report.py\')}}\n    signature = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()\n    output = args.output or args.root / \'results\' / f\'attention_{device.type}\'\n    output.mkdir(parents=True, exist_ok=True)\n    manifest_path, metrics_path = output / \'run_manifest.json\', output / \'metrics.json\'\n    if manifest_path.exists():\n        previous = json.loads(manifest_path.read_text(encoding=\'utf-8\'))\n        if previous[\'signature\'] != signature:\n            raise ValueError(\'Output belongs to a different experiment. Choose a new --output directory.\')\n        if metrics_path.exists():\n            prior = json.loads(metrics_path.read_text(encoding=\'utf-8\'))\n            if prior[\'complete\']:\n                write_reports(output, prior)\n                print((output / \'summary.md\').read_text(encoding=\'utf-8\'))\n                if not prior[\'passed\']:\n                    raise RuntimeError(\'This completed experiment has numerical failures; its artifacts were preserved.\')\n                return\n    write_json(manifest_path, {\'signature\': signature, \'identity\': identity})\n    payload = {\'complete\': False, \'passed\': False, \'device\': str(device), \'signature\': signature, \'cases\': []}\n    write_json(metrics_path, payload)\n    for index, (length, causal) in enumerate((n, causal) for n in config[\'lengths\'] for causal in config[\'causal_modes\']):\n        case = run_case(config, length, causal, device, config[\'seed\'] + index)\n        payload[\'cases\'].append(case)\n        write_json(metrics_path, payload)\n        print(case[\'case_id\'], \'passed:\', case[\'passed\'],\n              \'tiled max abs error:\', case[\'numerical\'][\'tiled\'][\'max_absolute_error\'], flush=True)\n    payload[\'complete\'] = len(payload[\'cases\']) == len(config[\'lengths\']) * len(config[\'causal_modes\'])\n    payload[\'passed\'] = payload[\'complete\'] and all(case[\'passed\'] for case in payload[\'cases\'])\n    write_json(metrics_path, payload)\n    write_reports(output, payload)\n    print(\'Results:\', output, flush=True)\n    if not payload[\'passed\']:\n        raise RuntimeError(\'Attention experiment retained one or more numerical/completeness failures.\')\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--root\', type=Path, default=ROOT)\n    parser.add_argument(\'--output\', type=Path)\n    parser.add_argument(\'--device\', choices=(\'auto\', \'cpu\', \'cuda\'), default=\'auto\')\n    run(parser.parse_args())\n', 'attention_benchmark_report.py': '"""Attention timing aggregation and scientific artifacts; no model dependencies."""\n\nimport csv\nimport math\nimport statistics\n\n\nMETHODS = (\'reference\', \'tiled\', \'sdpa_auto\')\n\n\ndef aggregate_measurements(records, *, repetitions):\n    expected = {(method, repeat) for method in METHODS for repeat in range(repetitions)}\n    seen = set()\n    for record in records:\n        key = (record[\'method\'], record[\'repeat\'])\n        if key not in expected or key in seen or not math.isfinite(record[\'seconds\']) or record[\'seconds\'] <= 0:\n            raise ValueError(\'Duplicate, unexpected or invalid timing measurement.\')\n        seen.add(key)\n    groups = {}\n    for method in METHODS:\n        rows = [r for r in records if r[\'method\'] == method]\n        if not rows:\n            continue\n        seconds = [r[\'seconds\'] for r in rows]\n        groups[method] = {\'repetitions\': len(rows), \'median_seconds\': statistics.median(seconds),\n                          \'min_seconds\': min(seconds), \'max_seconds\': max(seconds)}\n        for key in (\'initial_allocated_bytes\', \'peak_allocated_bytes\', \'peak_reserved_bytes\'):\n            values = [r[key] for r in rows if r[key] is not None]\n            groups[method][key] = max(values) if values else None\n    complete = seen == expected\n    return {\'complete\': complete, \'methods\': groups,\n            \'reference_over_tiled_speedup\': groups[\'reference\'][\'median_seconds\'] / groups[\'tiled\'][\'median_seconds\']\n            if complete else None}\n\n\ndef score_workspace_bytes(*, batch, heads, queries, keys, block_q, block_k, element_bytes):\n    return {\'kind\': \'theoretical_score_tensor_only\',\n            \'dense_score_tensor_bytes\': batch * heads * queries * keys * element_bytes,\n            \'largest_tiled_score_tensor_bytes\': batch * heads * min(block_q, queries) * min(block_k, keys) * element_bytes}\n\n\ndef write_reports(output, payload):\n    rows = payload[\'cases\']\n    fields = [\'case_id\', \'method\', \'passed\', \'max_absolute_error\', \'mean_absolute_error\',\n              \'median_seconds\', \'min_seconds\', \'max_seconds\', \'peak_allocated_bytes\', \'peak_reserved_bytes\']\n    with (output / \'benchmark.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields, extrasaction=\'ignore\')\n        writer.writeheader()\n        for case in rows:\n            for method, values in case[\'timing\'][\'methods\'].items():\n                writer.writerow({\'case_id\': case[\'case_id\'], \'method\': method,\n                                 **case[\'numerical\'][method], **values})\n    lines = [\'# Attention experiment\', \'\',\n        f"Device: **{payload[\'device\']}**; complete: **{payload[\'complete\']}**; numerical checks passed: **{payload[\'passed\']}**.", \'\',\n        \'| Case | Reference ms | Tiled ms | SDPA auto ms | Tiled max abs error | Reference / tiled |\',\n        \'|---|---:|---:|---:|---:|---:|\']\n    for case in rows:\n        timing = case[\'timing\'][\'methods\']\n        values = [timing[m][\'median_seconds\'] * 1000 for m in METHODS]\n        error = case[\'numerical\'][\'tiled\'][\'max_absolute_error\']\n        ratio = case[\'timing\'][\'reference_over_tiled_speedup\']\n        error_text = f\'{error:.3g}\' if error is not None else \'nonfinite\'\n        ratio_text = f\'{ratio:.3f}x\' if ratio is not None else \'unavailable\'\n        lines.append(f"| {case[\'case_id\']} | {values[0]:.3f} | {values[1]:.3f} | {values[2]:.3f} | {error_text} | {ratio_text} |")\n    lines += [\'\', \'Ratios below 1 mean the tiled implementation was slower. Every method uses the\',\n        \'same inputs and dtype. Times are medians; CSV/JSON retain min/max ranges and raw\',\n        \'repetitions. CUDA measurements synchronize before/after; warmups and transfers\',\n        \'are outside timing. Error is measured against a dense FP64 CPU oracle.\', \'\',\n        \'Custom implementations accumulate FP16/BF16 in FP32 and preserve FP64 inputs.\',\n        \'SDPA auto means PyTorch selected the backend; it is not a claim that a particular\',\n        \'fused kernel ran. This is standalone attention on controlled tensors, not model\',\n        \'accuracy, a training experiment, or a measured end-to-end Qwen speedup.\', \'\',\n        \'CPU peak tensor memory is not measured (null). Score-workspace sizes are\',\n        \'theoretical tensor sizes, not measured peak memory. GPU peaks include input\',\n        \'tensors and returned output plus intermediate allocations. Python loop overhead,\',\n        \'launches, temporary allocations and repeated tile operations can outweigh savings.\', \'\']\n    (output / \'summary.md\').write_text(\'\\n\'.join(lines), encoding=\'utf-8\')\n    if not rows:\n        return\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    figure, axes = plt.subplots(1, 2, figsize=(12, 4))\n    for method, offset in zip(METHODS, (-0.25, 0, 0.25)):\n        values = [r[\'timing\'][\'methods\'][method][\'median_seconds\'] * 1000 for r in rows]\n        lows = [(r[\'timing\'][\'methods\'][method][\'median_seconds\'] - r[\'timing\'][\'methods\'][method][\'min_seconds\']) * 1000 for r in rows]\n        highs = [(r[\'timing\'][\'methods\'][method][\'max_seconds\'] - r[\'timing\'][\'methods\'][method][\'median_seconds\']) * 1000 for r in rows]\n        axes[0].bar([i + offset for i in range(len(rows))], values, width=0.25,\n                    label=method, yerr=[lows, highs], capsize=2)\n        errors = [r[\'numerical\'][method][\'max_absolute_error\'] for r in rows]\n        axes[1].plot(range(len(rows)), [max(e, 1e-16) if e is not None else float(\'nan\') for e in errors],\n                     marker=\'o\', label=method)\n    for axis in axes:\n        axis.set_xticks(range(len(rows)), [r[\'case_id\'] for r in rows], rotation=45)\n        axis.legend()\n    axes[0].set_ylabel(\'Milliseconds (median; range bars)\')\n    axes[0].set_title(f"Attention runtime: {payload[\'device\']}")\n    axes[1].set_ylabel(\'Maximum absolute error vs FP64 oracle\')\n    axes[1].set_yscale(\'log\')\n    figure.tight_layout()\n    figure.savefig(output / \'attention_comparison.png\', dpi=160)\n    plt.close(figure)\n', 'test_attention.py': '"""Independent mathematical fixtures and real PyTorch attention comparisons."""\n\nimport unittest\n\nimport torch\nimport torch.nn.functional as F\nfrom torch.nn.attention import SDPBackend, sdpa_kernel\n\nfrom attention import reference_attention, tiled_attention\n\n\nclass AttentionTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        torch.set_num_threads(2)\n\n    def tensors(self, batch=1, heads=2, kv_heads=None, queries=7, keys=None,\n                dim=5, value_dim=3, dtype=torch.float64):\n        generator = torch.Generator().manual_seed(71)\n        keys = queries if keys is None else keys\n        kv_heads = heads if kv_heads is None else kv_heads\n        return tuple(torch.randn(shape, generator=generator, dtype=dtype) for shape in (\n            (batch, heads, queries, dim), (batch, kv_heads, keys, dim),\n            (batch, kv_heads, keys, value_dim)))\n\n    def independent(self, q, k, v, causal, offset):\n        mask = None\n        if causal:\n            mask = torch.arange(k.shape[2])[None, :] <= (torch.arange(q.shape[2])[:, None] + offset)\n        k = k.repeat_interleave(q.shape[1] // k.shape[1], dim=1)\n        v = v.repeat_interleave(q.shape[1] // v.shape[1], dim=1)\n        with sdpa_kernel(backends=[SDPBackend.MATH]):\n            return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0)\n\n    def test_hand_calculated_uniform_and_causal_values(self):\n        q = k = torch.zeros((1, 1, 2, 1), dtype=torch.float64)\n        v = torch.tensor([[[[2.0], [4.0]]]], dtype=torch.float64)\n        for function in (reference_attention, tiled_attention):\n            torch.testing.assert_close(function(q, k, v, causal=False),\n                torch.tensor([[[[3.0], [3.0]]]], dtype=torch.float64), atol=0, rtol=0)\n            torch.testing.assert_close(function(q, k, v, causal=True),\n                torch.tensor([[[[2.0], [3.0]]]], dtype=torch.float64), atol=0, rtol=0)\n\n    def test_multiple_shapes_offsets_and_gqa_match_independent_math(self):\n        for kwargs, offset in (({}, 0), ({\'batch\': 2, \'heads\': 4, \'queries\': 11}, 0),\n            ({\'heads\': 4, \'kv_heads\': 2, \'queries\': 13}, 0),\n            ({\'queries\': 1, \'keys\': 11}, 10), ({\'queries\': 3, \'keys\': 9}, 6)):\n            q, k, v = self.tensors(**kwargs)\n            for causal in (True, False):\n                expected = self.independent(q, k, v, causal, offset)\n                with self.subTest(shape=q.shape, causal=causal, offset=offset):\n                    torch.testing.assert_close(reference_attention(q, k, v, causal=causal, query_offset=offset), expected,\n                                               atol=1e-12, rtol=1e-12)\n                    for tile_q, tile_k in ((1, 1), (3, 4), (64, 64)):\n                        actual = tiled_attention(q, k, v, causal=causal, query_offset=offset,\n                                                 block_q=tile_q, block_k=tile_k)\n                        torch.testing.assert_close(actual, expected, atol=1e-12, rtol=1e-12)\n\n    def test_future_values_do_not_change_earlier_causal_outputs(self):\n        q, k, v = self.tensors(queries=9)\n        changed_k, changed_v = k.clone(), v.clone()\n        changed_k[:, :, 5:] *= 100\n        changed_v[:, :, 5:] += 10000\n        for function in (reference_attention, tiled_attention):\n            before = function(q, k, v, causal=True)\n            after = function(q, changed_k, changed_v, causal=True)\n            torch.testing.assert_close(before[:, :, :5], after[:, :, :5], atol=0, rtol=0)\n\n    def test_fully_masked_rows_are_zero_and_do_not_poison_later_rows(self):\n        q, k, v = self.tensors(queries=7, keys=9)\n        expected = self.independent(q, k, v, True, -3)\n        for function in (reference_attention, tiled_attention):\n            actual = function(q, k, v, causal=True, query_offset=-3)\n            self.assertTrue(torch.isfinite(actual).all())\n            self.assertEqual(torch.count_nonzero(actual[:, :, :3]).item(), 0)\n            torch.testing.assert_close(actual, expected, atol=1e-12, rtol=1e-12)\n\n    def test_later_larger_tile_maximum_rescales_old_accumulators(self):\n        q = torch.ones((1, 1, 1, 1), dtype=torch.float64)\n        k = torch.tensor([[[[-1000.0], [-1001.0], [1000.0], [1001.0]]]], dtype=torch.float64)\n        v = torch.tensor([[[[99.0], [98.0], [2.0], [4.0]]]], dtype=torch.float64)\n        # The first two weights underflow relative to the second tile\'s maximum.\n        expected = torch.tensor([[[[(2.0 + 4.0 * torch.exp(torch.tensor(1.0, dtype=torch.float64)).item()) /\n                                    (1.0 + torch.exp(torch.tensor(1.0, dtype=torch.float64)).item())]]]], dtype=torch.float64)\n        actual = tiled_attention(q, k, v, causal=False, block_q=1, block_k=2)\n        torch.testing.assert_close(actual, expected, atol=1e-12, rtol=1e-12)\n\n    def test_low_precision_uses_stable_accumulation_and_preserves_output_dtype(self):\n        for dtype, tolerance in ((torch.float32, 2e-5), (torch.float16, 2e-3), (torch.bfloat16, 2e-2)):\n            q, k, v = self.tensors(dtype=dtype, queries=15, dim=8)\n            expected = self.independent(q.double(), k.double(), v.double(), True, 0)\n            for function in (reference_attention, tiled_attention):\n                output = function(q, k, v, causal=True)\n                self.assertEqual(output.dtype, dtype)\n                torch.testing.assert_close(output.double(), expected, atol=tolerance, rtol=tolerance)\n        q = torch.full((1, 1, 4, 16), 1000.0, dtype=torch.float16)\n        k = q.clone()\n        v = torch.arange(4, dtype=torch.float16).reshape(1, 1, 4, 1)\n        for function in (reference_attention, tiled_attention):\n            output = function(q, k, v, causal=False)\n            self.assertTrue(torch.isfinite(output).all())\n            torch.testing.assert_close(output, torch.full_like(output, 1.5))\n\n    def test_noncontiguous_inputs_work_without_mutation(self):\n        q, k, v = self.tensors(queries=8, dim=8, value_dim=8)\n        q, k, v = (x.transpose(-1, -2) for x in (q, k, v))\n        before = [x.clone() for x in (q, k, v)]\n        expected = self.independent(q, k, v, True, 0)\n        for function in (reference_attention, tiled_attention):\n            torch.testing.assert_close(function(q, k, v), expected, atol=1e-12, rtol=1e-12)\n        for x, saved in zip((q, k, v), before):\n            self.assertTrue(torch.equal(x, saved))\n\n    def test_invalid_shapes_dtypes_devices_or_tile_sizes_fail(self):\n        q, k, v = self.tensors()\n        invalid = [(q[0], k, v), (q, k[:, :, :0], v[:, :, :0]),\n                   (q, k.float(), v), (q, k, v[:, :, :-1]),\n                   (q, k[:, :, :, :-1], v), (q.long(), k.long(), v.long()),\n                   (q, k.to(\'meta\'), v)]\n        for function in (reference_attention, tiled_attention):\n            for values in invalid:\n                with self.assertRaises(ValueError):\n                    function(*values)\n        for tile in (0, -1, True, 2.5):\n            with self.assertRaises(ValueError):\n                tiled_attention(q, k, v, block_q=tile)\n        with self.assertRaises(ValueError):\n            tiled_attention(*self.tensors(heads=3, kv_heads=2))\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'test_attention_benchmark.py': "import unittest\nimport json\nfrom pathlib import Path\nimport shutil\nimport tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\n\nimport torch\n\nfrom benchmark_attention import aggregate_measurements, error_metrics, score_workspace_bytes\n\n\nclass AttentionBenchmarkTests(unittest.TestCase):\n    def config(self):\n        return json.loads(Path(__file__).with_name('configs').joinpath('attention_config.json').read_text(encoding='utf-8'))\n\n    def test_empty_duplicate_or_bad_config_cannot_claim_success(self):\n        from benchmark_attention import validate_config\n        for changes in ({'lengths': []}, {'lengths': [8, 8]}, {'lengths': [0]},\n                        {'causal_modes': []}, {'causal_modes': [1]}, {'causal_modes': [True, True]},\n                        {'warmups': -1}, {'repetitions': 0}, {'atol': float('nan')}):\n            with self.assertRaises(ValueError):\n                validate_config({**self.config(), **changes})\n\n    def test_complete_metrics_can_regenerate_missing_reports_without_remeasurement(self):\n        import benchmark_attention\n        with tempfile.TemporaryDirectory() as folder:\n            root = Path(folder)\n            (root / 'configs').mkdir()\n            config = {**self.config(), 'lengths': [4], 'causal_modes': [True],\n                      'batch': 1, 'heads': 1, 'head_dim': 2, 'repetitions': 1, 'warmups': 0}\n            (root / 'configs/attention_config.json').write_text(json.dumps(config), encoding='utf-8')\n            for name in ('attention.py', 'benchmark_attention.py', 'attention_benchmark_report.py'):\n                shutil.copyfile(Path(__file__).with_name(name), root / name)\n            output = root / 'output'\n            arguments = SimpleNamespace(root=root, device='cpu', output=output)\n            benchmark_attention.run(arguments)\n            original = (output / 'metrics.json').read_bytes()\n            (output / 'summary.md').unlink()\n            with patch.object(benchmark_attention, 'run_case', side_effect=AssertionError('must not remeasure')):\n                benchmark_attention.run(arguments)\n            self.assertTrue((output / 'summary.md').is_file())\n            self.assertEqual((output / 'metrics.json').read_bytes(), original)\n\n    def records(self):\n        return [{'method': name, 'repeat': repeat, 'seconds': seconds,\n                 'initial_allocated_bytes': None, 'peak_allocated_bytes': None,\n                 'peak_reserved_bytes': None}\n                for name, samples in [('reference', [1, 2, 3]), ('tiled', [2, 4, 6]), ('sdpa_auto', [0.5, 1, 1.5])]\n                for repeat, seconds in enumerate(samples)]\n\n    def test_repeated_summary_retains_slowdown_ranges_and_missing_memory(self):\n        result = aggregate_measurements(self.records(), repetitions=3)\n        self.assertTrue(result['complete'])\n        self.assertEqual(result['methods']['reference']['median_seconds'], 2)\n        self.assertEqual(result['methods']['tiled']['min_seconds'], 2)\n        self.assertEqual(result['methods']['tiled']['max_seconds'], 6)\n        self.assertIsNone(result['methods']['tiled']['peak_allocated_bytes'])\n        self.assertEqual(result['reference_over_tiled_speedup'], 0.5)\n        self.assertFalse(aggregate_measurements(self.records()[:-1], repetitions=3)['complete'])\n\n    def test_duplicate_unexpected_or_invalid_measurements_fail(self):\n        rows = self.records()\n        for wrong in (rows + [rows[0]], [{**rows[0], 'method': 'unknown'}],\n                      [{**rows[0], 'seconds': float('nan')}], [{**rows[0], 'seconds': 0}]):\n            with self.assertRaises(ValueError):\n                aggregate_measurements(wrong, repetitions=3)\n\n    def test_error_metrics_report_real_errors_and_nonfinite_outputs_fail(self):\n        expected = torch.tensor([1.0, 2.0], dtype=torch.float64)\n        values = error_metrics(torch.tensor([1.125, 1.75]), expected, atol=0.01, rtol=0.01)\n        self.assertFalse(values['passed'])\n        self.assertEqual(values['max_absolute_error'], 0.25)\n        self.assertEqual(values['mean_absolute_error'], 0.1875)\n        bad = error_metrics(torch.tensor([float('nan'), 2.0]), expected, atol=1, rtol=1)\n        self.assertFalse(bad['passed'])\n        self.assertFalse(bad['finite'])\n        self.assertIsNone(bad['max_absolute_error'])\n\n    def test_theoretical_workspace_is_separate_from_peak_memory(self):\n        values = score_workspace_bytes(batch=1, heads=2, queries=128, keys=256,\n                                       block_q=16, block_k=32, element_bytes=4)\n        self.assertEqual(values['dense_score_tensor_bytes'], 262144)\n        self.assertEqual(values['largest_tiled_score_tensor_bytes'], 4096)\n        self.assertEqual(values['kind'], 'theoretical_score_tensor_only')\n\n\nif __name__ == '__main__':\n    unittest.main()\n", 'configs/attention_config.json': '{\n  "seed": 42,\n  "lengths": [128, 256, 512],\n  "causal_modes": [false, true],\n  "batch": 1,\n  "heads": 4,\n  "head_dim": 64,\n  "dtype": "float32",\n  "block_q": 64,\n  "block_k": 64,\n  "warmups": 2,\n  "repetitions": 5,\n  "cpu_threads": 2,\n  "atol": 0.00002,\n  "rtol": 0.0002\n}\n', 'ATTENTION.md': "# Task 4: tiled attention with online softmax\n\n## What is implemented\n\n`attention.py` supplies two inference-only implementations:\n\n- `reference_attention(q, k, v, causal=True, query_offset=0)` explicitly creates\n  scaled scores, applies a causal mask, normalizes with a stable softmax, and\n  computes weighted values.\n- `tiled_attention(..., block_q=64, block_k=64)` processes query/key blocks using\n  running maxima, normalizers and unnormalized weighted-value accumulators.\n  It does not call a fused attention API or allocate a full score matrix when\n  tiles are smaller than the sequence.\n\nInputs are finite tensors `[batch, heads, tokens, head_dim]` sharing device and\ndtype. Q/K feature dimensions match; V may have a different feature dimension.\nQ heads may be an integer multiple of KV heads for grouped-query attention.\nQuery i has absolute position `i + query_offset`; keys have positions `0..S-1`.\nFor a cached last-token query over S keys, pass `query_offset=S-1`. The default\noffset is zero, including for rectangular shapes; it is not inferred silently.\nFully masked rows return zeros. FP16/BF16 inputs accumulate in FP32; FP64 is\npreserved. Output dtype equals the input dtype. No dropout, bias, padding mask,\nbackward implementation or nonfinite-input guarantee is supplied.\n\n## Algorithm and numerical stability\n\nFor query block Q and key/value block K,V, form `S = Q K^T / sqrt(d)` and mask\nfuture keys to negative infinity. Each query row maintains running maximum m,\ndenominator l, and value numerator a, initialized to `-inf`, zero and zero:\n\n```text\nm_new = max(m, rowmax(S))\nalpha = exp(m - m_new)\nP = exp(S - m_new)\nl_new = alpha * l + rowsum(P)\na_new = alpha * a + P @ V\nm, l, a = m_new, l_new, a_new\noutput = a / l\n```\n\nWhen both the old and new maxima are `-inf`, subtraction would produce NaN.\nThe implementation substitutes a safe zero maximum for that calculation,\nmaking masked probabilities/rescaling zero and retaining a zero accumulator.\nFinal zero denominators use one for division, yielding zero output.\nWhen a later tile has a larger maximum, rescaling both old l and old a is\nessential; otherwise probabilities from different tiles use incompatible scales.\nEach partial final tile uses its actual dimensions. Entire future-only key tiles\nare skipped, while boundary tiles receive elementwise causal masking.\n\nThe dense score tensor has `B * Hq * L * S` elements. The largest tiled score\ntensor has at most `B * Hq * block_q * block_k` elements, plus linear input,\noutput, conversion and accumulator storage. GQA expansion happens per key/value\ntile. Several tile intermediates may coexist, so one tile's byte count is not\ntotal peak memory. Python/PyTorch allocations are not guaranteed to stay in\non-chip SRAM; this educational implementation is not a custom fused CUDA kernel.\n\n## Validation\n\nUnit tests include literal hand-calculated attention values; independent PyTorch\nmath-SDPA comparisons; causal future-token changes; negative offsets and all-masked\nrows; rectangular cached queries; GQA; noncontiguous inputs; partial/unit/large\ntiles; later larger maxima; high FP16 magnitudes; FP32/FP64/BF16/FP16; and bad\nshapes, devices and dtypes. No model weights or dataset are needed.\n\nThe benchmark compares each implementation with a dense FP64 CPU oracle using\nthe exact same input values. Default FP32 tolerances are `atol=2e-5, rtol=2e-4`.\nMaximum/mean absolute error, maximum relative error, finiteness and pass flags\nare saved for every case/method. Relative error can be large near zero even\nwhen absolute error is small; the declared absolute-plus-relative tolerance\ndetermines passing. Library SDPA is a comparator, not used inside custom code.\n\n## Performance experiment\n\n`configs/attention_config.json` defines lengths 128/256/512, full and causal\nattention, batch one, four heads, head dimension 64, FP32, tiles 64x64, two\nwarmups, five repetitions and two CPU threads. There are six cases, three methods\nand **90 timed calls**. Methods rotate order across repetitions. Inputs are\ngenerated once per case on CPU with a recorded seed, then transferred outside\ntiming. Highest FP32 matmul precision is selected and recorded.\n\nMethods: dense reference, tiled online softmax, and `sdpa_auto` (PyTorch's selected\nbackend, without claiming any specific fused kernel). CUDA calls synchronize\nbefore and after timing. Garbage collection, allocator cache clearing, warmup,\nCPU-oracle work, device transfers, reporting and plotting are outside timing.\nKernel launches and temporary allocations inside each attention function count.\nClearing unused GPU allocator blocks gives a warmed-kernel, cleared-pool protocol;\nit is not an end-to-end serving benchmark.\n\nGPU allocated/reserved peaks include resident input tensors, output and temporary\nallocations. Initial allocated bytes are recorded. Native CPU tensor peak memory\nis **not measured**, so CPU memory fields are null. The separately labelled\nscore-workspace estimates are analytical tensor sizes, not measured process RAM\nor a claim of total peak savings. Ratios below one are slowdowns and are retained.\n\nPython loops, repeated operations, launches and allocation overhead can make the\neducational tiled path slower than dense operations or optimized SDPA. Standalone\nattention results do not establish an end-to-end Qwen improvement. This module\ndoes not replace the selected model's attention implementation or alter its\nclinical outputs; the later browser benchmark view can expose the experiment.\n\n## Run locally or in Colab\n\n```powershell\npython -m unittest -v test_attention test_attention_benchmark\npython benchmark_attention.py --device cpu --output results/attention_cpu_final\npython make_colab_attention.py\n```\n\nOpen `colab_attention.ipynb` in Colab and run its three steps. Source files are\nembedded and checksummed in the notebook: **no ZIP, dataset, adapter or model\nupload is required**. It uses a GPU when available, otherwise CPU, and downloads\n`attention_results.zip` at the end. It does not train or load Qwen weights.\nThe result ZIP contains manifest/configuration, numerical results, raw repeated\ntimings, CSV, summary and a plot. Return it for measured GPU analysis.\n\nThe output folder refuses a different source/configuration/environment identity.\nA matching completed experiment regenerates its reports and is displayed without\nrepeating measurements; recorded failures\nare not replaced by a rerun. An interrupted incomplete run restarts the small\nexperiment. Choose a new output directory to intentionally run another version.\n\n## Sources\n\n- [FlashAttention: exact attention with tiling and online normalization](https://arxiv.org/html/2205.14135v2)\n- [Milakov and Gimelshein: online softmax normalizer](https://arxiv.org/html/1805.02867v2)\n- [PyTorch SDPA scaling, masks and backend behavior](https://docs.pytorch.org/docs/2.15/generated/torch.nn.functional.scaled_dot_product_attention.html)\n\nThe equations follow these primary sources. The paper's optimized-kernel speedups\nare not attributed to this Python implementation.\n\n## Completed results\n\nVerified CPU results are in `results/attention_cpu_final/`; verified Tesla T4 GPU\nresults are in `results/attention_gpu/`. Both six-case experiments passed, with\n90 timed calls each. Largest tiled absolute errors versus FP64 were approximately\n6.91e-7 on both devices. The tiled path is slower in all measured cases; at length\n512 it uses about 51.5% less peak allocated GPU memory than the dense reference.\nFull measured tables, verification limits and interpretation are recorded in\n`reports/attention_analysis.md`. Task 4 is complete; no model retraining is needed.\n"}
EXPECTED_SHA256 = {'attention.py': 'df22dc21f739b9e601af444e0dff2b224552db35e367ce6bb4cc8d2a8c43d72a', 'benchmark_attention.py': 'f3e4d9da892feeee4e058f653c8b9ae3e4eb64c3314613b60c5d1d61705e3b78', 'attention_benchmark_report.py': 'ccfa3b38471e491748034d6a9dd6af81005239f6baa8ffcca8709f4c0e364db0', 'test_attention.py': '600cc893320a4eeb439bb339b08bc76b133cea98055700a51fd3d3fbe1d70830', 'test_attention_benchmark.py': '6f94d211f93160faea4c6246815358c33c8ecc85d7e9f9181c9e98848a5fea93', 'configs/attention_config.json': 'ad41ce19a498226973e027d00308e34b26356f29d8f68320e1da3358e96ba106', 'ATTENTION.md': '81a9ae3d8861d18dbf9b8d65f927b35d0caceb563ad5897a7a2d0270d35c585a'}
for name, source in SOURCES.items():
    data = source.encode("utf-8")
    if hashlib.sha256(data).hexdigest() != EXPECTED_SHA256[name]:
        raise ValueError(f"Embedded source checksum mismatch: {name}")
    target = PROJECT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(data)
print("Verified", len(SOURCES), "embedded files. No uploads required.")


## 2. Run correctness tests and the attention benchmark

Inputs are controlled random tensors, not patient records. The experiment
compares full and causal attention at lengths 128, 256 and 512, using two
warmups and five repetitions per method. It measures numerical error against
FP64, runtime and GPU memory where available. It does not rerun KV decoding.


In [ ]:
import importlib.util
import subprocess
import sys
import torch

def run_checked(arguments):
    with subprocess.Popen(arguments, cwd=PROJECT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    if code:
        raise RuntimeError(f"Command failed ({code}); the full error is printed above.")

if importlib.util.find_spec("matplotlib") is None:
    run_checked([sys.executable, "-m", "pip", "install", "--quiet", "matplotlib==3.10.7"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Benchmark device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
RUN = Path("/content/attention_results")
run_checked([sys.executable, "-m", "unittest", "-v", "test_attention", "test_attention_benchmark"])
run_checked([sys.executable, "-u", "benchmark_attention.py", "--device", DEVICE, "--output", str(RUN)])
print((RUN / "summary.md").read_text(encoding="utf-8"))


## 3. Inspect the plot and download attention_results.zip


In [ ]:
import zipfile
from google.colab import files
from IPython.display import Image, display

if (RUN / "attention_comparison.png").is_file():
    display(Image(filename=str(RUN / "attention_comparison.png")))
if not RUN.is_dir():
    raise FileNotFoundError("The benchmark has not created its output folder yet.")
download = Path("/content/attention_results.zip")
with zipfile.ZipFile(download, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in RUN.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(RUN).as_posix())
files.download(str(download))
